# Syria NW3 Population Zonal Statistics Map (2026)

Overview

Estimates the 2026 population of Aleppo, Idleb and Lattakia governorates using WorldPop raster data and zonal statistics. Population values are calculated by summing source raster cells whose centres fall within each governorate boundary. The results are displayed using area-proportional circles on an interactive dark basemap.

WorldPop 2026の人口ラスタとZonal Statisticsを用いて、Aleppo、Idleb、Lattakiaの3県における推計人口を算出します。各県の行政界内に中心点が含まれる元ラスタセルの人口値を合計し、県別人口を計算します。集計結果は、円の面積が人口に比例するシンボルとしてダークベースマップ上に表示します。

Objectives

- Select Aleppo, Idleb and Lattakia governorates
- Validate the administrative boundaries and population raster
- Calculate estimated population using zonal statistics
- Apply the cell-centre rule to boundary pixels
- Display governorate population using area-proportional circles
- Add administrative boundaries, labels and a numeric legend
- Export the completed map as an HTML file

- Aleppo、Idleb、Lattakiaの3県を解析対象として選択する
- 行政界と人口ラスタを検証する
- Zonal Statisticsを用いて県別推計人口を算出する
- 境界画素に画素中心方式を適用する
- 円の面積が人口に比例するシンボルで県別人口を表示する
- 行政界、ラベル、数値付き比例円凡例を追加する
- 完成した地図をHTMLファイルとして保存する

Workflow

1. Define the input and output paths
2. Read the administrative boundaries and population raster metadata
3. Validate the coordinate reference systems and NoData value
4. Select and validate the three target governorates
5. Calculate population totals using zonal statistics
6. Prepare representative points and population values
7. Calculate area-proportional circle radii
8. Create a dark basemap without built-in place labels
9. Add administrative boundaries, labels and proportional circles
10. Add the information panel, numeric legend and layer control
11. Save and display the interactive map

Data

Population raster data:

- `worldpop_syria_2026.tif`

Source: WorldPop, 2026 estimated population dataset

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Data Scope and Limitations

- Population values are estimates rather than census counts.
- Only Aleppo, Idleb and Lattakia governorates are included.
- The source raster resolution is 3 arc seconds, approximately 100 metres.
- Population totals are calculated by summing the source grid-cell values.
- `all_touched=False` includes cells whose centres fall within each governorate boundary.
- Boundary cells whose centres fall outside a governorate are not included.
- Circle locations are representative points within the governorate polygons and do not indicate governorate capitals.
- Circle areas, rather than circle radii, are proportional to estimated population.
- Results depend on the source raster, administrative boundaries and the selected boundary-cell rule.

- 人口値は国勢調査による実測値ではなく推計値です。
- 解析対象はAleppo、Idleb、Lattakiaの3県に限定しています。
- 元ラスタの解像度は3秒角で、約100メートルです。
- 人口総数は元ラスタのグリッドセル値を合計して算出します。
- `all_touched=False`により、中心点が各県の行政界内にあるセルを集計します。
- 中心点が行政界外にある境界セルは集計に含まれません。
- 円の配置地点は県ポリゴン内の代表点であり、県都の位置を示すものではありません。
- 円の半径ではなく、円の面積が推計人口に比例します。
- 結果は元ラスタ、行政界、境界セルの集計方式に依存します。

Technologies

- Python
- GeoPandas
- Rasterio
- rasterstats
- Folium
- pathlib

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from math import isfinite
from pathlib import Path

import folium
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

In [ ]:
# 2
# Define the input and output paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
RASTER_DIR = ROOT_DIR / "02_DATA" / "RASTER"

admin0_path = VECTOR_DIR / "syr_admin0.geojson"
admin1_path = VECTOR_DIR / "syr_admin1.geojson"

population_path = RASTER_DIR / "worldpop_syria_2026.tif"

output_path = PROJECT_DIR / "03_syria_zonal_statistics.html"

In [ ]:
# 3
# Read and validate the spatial datasets
# 空間データを読み込み、座標参照系とNoData値を検証する

admin0 = gpd.read_file(
    admin0_path
)

admin1 = gpd.read_file(
    admin1_path
)

with rasterio.open(
    population_path
) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    population_nodata = src.nodata
    raster_band_count = src.count

datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in datasets.items():
    if dataset.empty:
        raise ValueError(
            f"{dataset_name} contains no features."
        )

    if dataset.crs is None:
        raise ValueError(
            f"{dataset_name} has no defined CRS."
        )

if raster_crs is None:
    raise ValueError(
        "The population raster has no defined CRS."
    )

if population_nodata is None:
    raise ValueError(
        "The population raster has no defined NoData value."
    )

if raster_band_count < 1:
    raise ValueError(
        "The population raster contains no readable bands."
    )

if admin0.crs != admin1.crs:
    raise ValueError(
        "Country and governorate boundaries "
        "must use the same CRS."
    )

if admin1.crs != raster_crs:
    raise ValueError(
        "Governorate boundaries and the population raster "
        "must use the same CRS. "
        f"Vector CRS: {admin1.crs}; "
        f"Raster CRS: {raster_crs}"
    )

print(
    f"Country boundary CRS: {admin0.crs}"
)

print(
    f"Governorate boundary CRS: {admin1.crs}"
)

print(
    f"Population raster CRS: {raster_crs}"
)

print(
    f"Population raster NoData: {population_nodata}"
)

print(
    f"Population raster bounds: {raster_bounds}"
)

In [ ]:
# 4
# Select and validate the target governorates
# 解析対象となる3県を選択し、内容を確認する

TARGET_GOVERNORATES = {
    "Aleppo",
    "Idleb",
    "Lattakia",
}

required_columns = {
    "adm1_name",
    "adm1_pcode",
    "geometry",
}

missing_columns = (
    required_columns
    - set(admin1.columns)
)

if missing_columns:
    raise ValueError(
        "The governorate dataset is missing required columns: "
        f"{sorted(missing_columns)}"
    )

target_admins = (
    admin1[
        admin1["adm1_name"].isin(
            TARGET_GOVERNORATES
        )
    ]
    .copy()
    .sort_values(
        "adm1_name"
    )
    .reset_index(
        drop=True
    )
)

selected_governorates = set(
    target_admins["adm1_name"]
)

missing_governorates = (
    TARGET_GOVERNORATES
    - selected_governorates
)

if missing_governorates:
    raise ValueError(
        "The following target governorates were not found: "
        f"{sorted(missing_governorates)}"
    )

if target_admins.geometry.isna().any():
    raise ValueError(
        "One or more target governorates have missing geometries."
    )

if target_admins.geometry.is_empty.any():
    raise ValueError(
        "One or more target governorates have empty geometries."
    )

if not target_admins.geometry.is_valid.all():
    raise ValueError(
        "One or more target governorates have invalid geometries."
    )

print(
    target_admins[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ]
)

In [ ]:
# 5
# Calculate population totals with zonal statistics
# Zonal Statisticsを用いて3県の推計人口を集計する

stats = zonal_stats(
    vectors=target_admins,
    raster=population_path,
    band=1,
    stats=[
        "sum",
    ],
    nodata=population_nodata,
    all_touched=False,
    geojson_out=False,
)

In [ ]:
# 6
# Prepare and validate the population results
# 県別人口の集計結果を検証し、地図表示用に整理する

if len(stats) != len(target_admins):
    raise ValueError(
        "The number of zonal-statistics results "
        "does not match the number of target governorates."
    )

population_results = []

for (_, governorate), result in zip(
    target_admins.iterrows(),
    stats,
):
    population = result.get(
        "sum"
    )

    if population is None:
        raise ValueError(
            "No population total was calculated for "
            f"{governorate['adm1_name']}."
        )

    population = float(
        population
    )

    if not isfinite(population):
        raise ValueError(
            "The calculated population is not finite for "
            f"{governorate['adm1_name']}."
        )

    if population < 0:
        raise ValueError(
            "The calculated population is negative for "
            f"{governorate['adm1_name']}."
        )

    representative_point = (
        governorate.geometry.representative_point()
    )

    population_results.append(
        {
            "name": governorate["adm1_name"],
            "pcode": governorate["adm1_pcode"],
            "population": population,
            "lat": representative_point.y,
            "lon": representative_point.x,
        }
    )

print(
    "Governorate population results:"
)

for item in population_results:
    print(
        f"{item['name']} ({item['pcode']}): "
        f"{item['population']:,.0f}"
    )

In [ ]:
# 7
# Calculate area-proportional circle radii
# 円の面積が人口に比例する半径を計算する

MAX_RADIUS = 28

if not population_results:
    raise ValueError(
        "No population results are available "
        "for proportional-circle calculation."
    )

max_population = max(
    item["population"]
    for item in population_results
)

if max_population <= 0:
    raise ValueError(
        "The maximum population must be greater than zero."
    )


def calculate_proportional_radius(population):
    """
    Calculate a circle radius whose area is proportional
    to the population value.

    円の面積が人口値に比例する半径を計算する。
    """

    return MAX_RADIUS * (
        population / max_population
    ) ** 0.5


for item in population_results:
    item["radius"] = calculate_proportional_radius(
        item["population"]
    )

print(
    "Proportional-circle radii:"
)

for item in population_results:
    print(
        f"{item['name']}: "
        f"{item['radius']:.2f} pixels"
    )

In [ ]:
# 8
# Create a dark basemap focused on the target governorates
# 対象3県を中心とした地名表記のない暗色ベースマップを作成する

m = folium.Map(
    location=[35.9, 36.8],
    zoom_start=7,
    tiles=None,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "dark_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    ),
    name="CARTO Dark — No Labels",
    overlay=False,
    control=True,
).add_to(
    m
)

# Calculate the combined bounds of the three governorates.
# 対象3県を合わせた地理的範囲を取得する

min_x, min_y, max_x, max_y = (
    target_admins.total_bounds
)

target_bounds = [
    [min_y, min_x],
    [max_y, max_x],
]

# Fit the initial view to the three governorates.
# HTMLを開いた際の初期表示を対象3県へ合わせる

m.fit_bounds(
    target_bounds,
    padding=(40, 40),
    max_zoom=8,
)

In [ ]:
# 9
# Add the country and governorate boundaries
# 国境および県境レイヤーを追加する

folium.GeoJson(
    str(admin0_path),
    name="Country Boundary",
    style_function=lambda feature: {
        "color": "white",
        "weight": 3,
        "fillOpacity": 0,
    },
).add_to(
    m
)

folium.GeoJson(
    str(admin1_path),
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "color": "gray",
        "weight": 1,
        "fillOpacity": 0,
    },
).add_to(
    m
)

In [ ]:
# 10
# Add neighbouring country labels
# 周辺国名を独立したラベルレイヤーとして追加する

neighbors = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0],
}

neighbor_label_group = folium.FeatureGroup(
    name="Neighbour Labels",
    overlay=True,
    control=True,
    show=True,
)

for name, coordinates in neighbors.items():
    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 14pt;
                font-weight: bold;
                color: lightgray;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        ),
    ).add_to(
        neighbor_label_group
    )

neighbor_label_group.add_to(
    m
)

In [ ]:
# 11
# Prepare a reduced target-governorate layer
# 地図表示に必要な県名とジオメトリだけを抽出する

target_admins_map = target_admins[
    [
        "adm1_name",
        "geometry",
    ]
].copy()

In [ ]:
# 12
# Add the target-governorate layer
# 解析対象となる3県のレイヤーを追加する

folium.GeoJson(
    target_admins_map,
    name="Target Governorates",
    style_function=lambda feature: {
        "fillColor": "#b7d86f",
        "fillOpacity": 0.15,
        "color": "white",
        "weight": 1,
    },
    highlight_function=lambda feature: {
        "fillOpacity": 0.25,
        "color": "#b7d86f",
        "weight": 2,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
        ],
        aliases=[
            "Governorate:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

In [ ]:
# 13
# Add the proportional-circle population layer
# 県別推計人口を示す比例円レイヤーを追加する

population_circle_group = folium.FeatureGroup(
    name="Estimated Population Circles",
    overlay=True,
    control=True,
    show=True,
)

for item in population_results:
    folium.CircleMarker(
        location=[
            item["lat"],
            item["lon"],
        ],
        radius=item["radius"],
        color="#b7d86f",
        weight=1,
        fill=True,
        fill_color="#b7d86f",
        fill_opacity=0.7,
        popup=folium.Popup(
            (
                f"<b>{item['name']}</b><br>"
                f"Pcode: {item['pcode']}<br>"
                f"Estimated population: "
                f"{item['population']:,.0f}"
            ),
            max_width=260,
        ),
    ).add_to(
        population_circle_group
    )

population_circle_group.add_to(
    m
)

In [ ]:
# 14
# Add governorate labels
# シリア14県の名称を独立したラベルレイヤーとして追加する

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1],
}

governorate_label_group = folium.FeatureGroup(
    name="Governorate Labels",
    overlay=True,
    control=True,
    show=True,
)

for name, coordinates in governorates.items():
    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 10pt;
                color: white;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        ),
    ).add_to(
        governorate_label_group
    )

governorate_label_group.add_to(
    m
)

In [ ]:
# 15
# Add population value labels
# 比例円上に県別推計人口を表示するラベルレイヤーを追加する

population_value_group = folium.FeatureGroup(
    name="Population Value Labels",
    overlay=True,
    control=True,
    show=True,
)

for item in population_results:
    folium.Marker(
        location=[
            item["lat"],
            item["lon"],
        ],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                color: white;
                font-size: 12pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                text-shadow: 0 0 4px black;
            ">
                {item["population"] / 1_000_000:.2f} M
            </div>
            """
        ),
    ).add_to(
        population_value_group
    )

population_value_group.add_to(
    m
)

In [ ]:
# 16
# Add the map information panel
# 地図の概要、集計方法、出典を示す情報パネルを追加する

title_html = """
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 390px;
    background-color: rgba(0, 0, 0, 0.75);
    color: white;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid white;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.5);
">
    <b style="font-size: 16px;">
        Northwest Syria
    </b>
    <br>

    <span style="
        color: #b7d86f;
        font-weight: bold;
    ">
        Estimated Population by Governorate (2026)
    </span>

    <small style="
        display: block;
        margin-top: 6px;
        line-height: 1.35;
        color: #eeeeee;
    ">
        Population totals for Aleppo, Idleb and Lattakia
        were calculated from the WorldPop 2026 raster
        using zonal statistics. Source grid cells whose
        centres fall within each governorate boundary
        are included. Circle area is proportional to the
        estimated population.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 6px;
        font-size: 11px;
        line-height: 1.35;
        color: #bbbbbb;
        border-top: 1px solid #444444;
    ">
        Population source:
        <a
            href="https://hub.worldpop.org/geodata/summary?id=75632"
            target="_blank"
            style="
                color: #3498db;
                text-decoration: none;
            "
        >
            WorldPop 2026
        </a>
        <br>
        Boundary source: HDX OCHA
        <br>
        Method: Zonal Statistics / Cell-Centre Rule /
        Proportional-Circle Map
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        title_html
    )
)

In [ ]:
# 17
# Add the numeric proportional-circle legend
# 人口値と円の大きさを対応させた比例円凡例を追加する

legend_reference_values = [
    5_000_000,
    3_000_000,
    1_000_000,
]

legend_rows = []

for population_value in legend_reference_values:
    diameter = (
        calculate_proportional_radius(
            population_value
        )
        * 2
    )

    row_height = max(
        diameter + 8,
        34,
    )

    legend_rows.append(
        f"""
        <div style="
            display: flex;
            align-items: center;
            height: {row_height:.1f}px;
        ">
            <span style="
                display: inline-block;
                width: {diameter:.1f}px;
                height: {diameter:.1f}px;
                min-width: {diameter:.1f}px;
                background-color: rgba(183, 216, 111, 0.7);
                border: 1px solid #b7d86f;
                border-radius: 50%;
                margin-right: 12px;
            "></span>

            <span>
                {population_value / 1_000_000:.0f} million
            </span>
        </div>
        """
    )

legend_rows_html = "".join(
    legend_rows
)

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    right: 40px;
    width: 245px;
    background-color: rgba(0, 0, 0, 0.78);
    color: white;
    border: 1px solid white;
    border-radius: 8px;
    padding: 12px;
    font-size: 12px;
    z-index: 9999;
">
    <b style="font-size: 13px;">
        Estimated Population
    </b>

    <div style="
        margin-top: 12px;
    ">
        {legend_rows_html}
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        color: #bbbbbb;
        border-top: 1px solid #444444;
        line-height: 1.3;
    ">
        Circle area is proportional to population.
        <br>
        All circles use the same population scale.
        <br>
        Source: WorldPop 2026
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        legend_html
    )
)

In [ ]:
# 18
# Add the layer control
# 地図レイヤーの表示と非表示を切り替える機能を追加する

folium.LayerControl(
    collapsed=False,
).add_to(
    m
)

In [ ]:
# 19
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m